# This notebook describes the workflow for creating figures describing the temporal trends of activity calculations saved as json files per person with averages over their followup

## Set up your environment
You will want to set your Cloud compute profile to have: 16 CPUs, 104GB RAM, 500GB Disk and the Compute type should be a highmem to have that configuration


In [ ]:
## Import libraries
import glob
import numpy as np
import h5py
import pandas as pd
from google.cloud import storage
import json
import time
import os
import subprocess
import matplotlib.pyplot as plt
from collections import defaultdict
from datetime import datetime, timedelta
import math
from tqdm.notebook import tqdm
from joblib import Parallel, delayed
import multiprocessing

In [ ]:

os.environ["WORKSPACE_CDR"] = #specify Google Project
os.environ["WORKSPACE_BUCKET"] = #specify GCP bucket

bucket_= # specify local fuse mounted path
BUCKET = # specify gs bucket path

#paths that are hardcoded to save files generated
file_paths_values = pd.read_csv('filepaths_values.csv') # load your version(loading the exmaple version share here, you can renmae based on your preferences)

file_paths_values = file_paths_values[file_paths_values['notebook']=='trends_figures'][['notebook','key_','value_']]
paths_dict = file_paths_values.set_index("key_")["value_"]

## Some custom functions used later


In [ ]:
def list_json_files(bucket_name, folders,prefix_name):
    """
    Lists all JSON files from specified folders in a Google Cloud Storage bucket.
    
    Parameters:
    -----------
    bucket_name : str
        Name of the Google Cloud Storage bucket    
    folders : list
        List of folder names to search within the bucket
        Example: ["jsons_calculations_1", "jsons_calculations_2", "jsons_calculations_3"]   
    prefix_name : str
        Base prefix/directory name that precedes the folder structure
        Example: "trends_activity"    
    Returns:
    --------
    list
        List of full blob names (paths) for all JSON files found
        Example: ["trends_activity_level_second_level_data/jsons_calculations_1/12345.json", ...]
    """
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    json_files = []

    for folder in folders:
        prefix = f"{prefix_name}/{folder}/"  # Ensure correct folder structure
        blobs = bucket.list_blobs(prefix=prefix)

        for blob in blobs:
            if blob.name.endswith('.json'):
                json_files.append(blob.name)

    return json_files

def read_json_file(bucket_name, file_name):
    """
     Reads and parses a JSON file from Google Cloud Storage.
    
    Parameters:
    -----------
    bucket_name : str
        Name of the Google Cloud Storage bucket
        
    file_name : str
        Full path/name of the JSON file within the bucket
        Example: "trends_activity_level_second_level_data/jsons_calculations_1/12345.json"
    
    Returns:
    --------
    dict
        Parsed JSON content
        
    """
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(file_name)

    content = blob.download_as_text()
    return json.loads(content)



In [ ]:
def get_ylabel(key, kk):
    """
    Smart ylabel generator based on activity key and time period for plot labeling.
    
    Parameters:
    -----------
    key : str
        Activity type key (e.g., 'activity minutes', 'steps')
        
    kk : str
        Time period type ('Daily', 'Weekly', 'Monthly', 'Yearly')
        Affects whether to show 'Proportion of' (Daily) vs 'Average' (other periods)
    Returns:
    --------
    str
        Formatted y-axis label for plots
    """        
    ylabel_map = {
        'steps': "Average Step count",
        'mvg': f"{'Proportion of' if kk == 'Daily' else 'Average'} moderate to vigorous minutes",
        'sd': f"{'Proportion of' if kk == 'Daily' else 'Average'} sedentary minutes",
        'lact': f"{'Proportion of' if kk == 'Daily' else 'Average'} light activity minutes"
    }
    
    
    label = next((ylabel_map[k] for k in ylabel_map if k in key))
    return label

def get_ycutoff(key, kk):
    """
    Smart y-axis cutoff generator for consistent plot scaling across activity types.
    
    Parameters:
    -----------
    key : str
        Activity type key (e.g., 'activity minutes', 'steps')
        
    kk : str
        Time period type (currently unused but maintained for API consistency)
    
    Returns:
    --------
    tuple
        (minimum_cutoff, maximum_cutoff) for y-axis scaling
    """
    mfactor_map = {
        'steps': 1000,
        'mvg': 10,
        'sd': 100,
        'lact': 120,
    }
    afactor_map = {
        'steps': 3000,
        'mvg': 20,
        'sd': 300,
        'lact':120,
    }
    
    fst = next((mfactor_map[k]*10 if ('limited' in key and 'sd' in key) else mfactor_map[k] for k in mfactor_map if k in key))
    sec = next((afactor_map[k] for k in afactor_map if k in key))
    return fst,sec

def get_ycutoff_line(key, kk):
    """Smart ylabel generator based on key and time period"""
    mfactor_map = {
        'steps': 1000,
        'hr': 10,
        'mvg': 10,
        'sd': 100,
        'lact': 120,
    }
    afactor_map = {
        'steps': 300,
        'hr': 30,
        'mvg': 2,
        'sd': 5,
        'lact':120,
    }
    
    fst = next((mfactor_map[k]*10 if ('limited' in key and 'sd' in key) else mfactor_map[k] for k in mfactor_map if k in key))
    sec = next((afactor_map[k] for k in afactor_map if k in key))
    return fst,sec

In [ ]:
# DataFrame containing heart rate data followup estimates:
# - 'person_id': participant identifier
# - 'longest_streak_start': start date of longest continuous data period
# - 'longest_streak_end': end date of longest continuous data period
# - 'longest_relaxed_streak': months of followup in the streak
# - 'first_active_year': year when first active

df_summary_hr = pd.read_csv(f'{bucket_}/{paths_dict.get("followup_heartrate_file")}.tsv',sep='\t')
df_summary_hr['person_id'] = df_summary_hr['person_id'].astype(int)

#select individuals with atleast 12 months of data, n beginning before 2023 as v8 stops in oct 2023 (so wont be complete year)
seasonal_hr = list(set(df_summary_hr[(df_summary_hr['first_active_year']<2023)&\
          (df_summary_hr['longest_relaxed_streak']>=12)]['person_id']))

#select individuals with 3 full years of data of data, n beginning before 2023
df_3_yearly_hr = list(set(df_summary_hr[(df_summary_hr['first_active_year']<2023)&\
          (df_summary_hr['longest_relaxed_streak']>=12*3)]['person_id']))
#select individuals with 5 full years of data of data, n beginning before 2023
df_5_yearly_hr = list(set(df_summary_hr[(df_summary_hr['first_active_year']<2023)&\
          (df_summary_hr['longest_relaxed_streak']>=12*5)]['person_id']))


## Person level aggregates

This code implements a parallel processing pipeline to extract and aggregate individual-level activity summary statistics from thousands of JSON files in one file. It processes wearable device data to create a consolidated dataset where each participant has a single row containing their average activity metrics across all available data calculated earlier, enabling participant-level analysis

In [ ]:
# #copy the files locally to speed it up
subprocess.run(f'gcloud storage cp -r {bucket_}/{paths_dict.get("trends_save")} .', shell=True, check=True)

In [ ]:
#Helper Function - JSON File Reader
def read_json_local_file(name):
    with open(name, "r") as json_file:
        step_Data = json.load(json_file)
    return step_Data


def save_mvg_agg(file_n,keys = ['steps_limited','mvg_limited',\
                                    'sd_limited','lact_limited'],metric = 'non_nan_avg'):
    
    """
    Parameters:
    
    file_n: Path to individual participant's JSON file
    keys: List of activity metrics to extract
    metric: Statistical measure to use (default: 'non_nan_avg')
    """
    
    #since we already have a calculated average value across all person_Days, which extract that
    data = []
    step_data = read_json_local_file(file_n)
    dict_ = {}
    dict_['person_id'] = int(file_n.split('/')[-1].split('.')[0])
    for key in keys:
        dict_[f'avg_{key}'] = step_data[f"daily_one_value_act_lvl"][key][metric]
    try:
        data.append(dict_)
    except KeyError as e:
        print(f"KeyError: {e} in file {file_n}")


    df_mvg_agg = pd.DataFrame(data)
    return df_mvg_agg

keys = ['steps_limited','mvg_limited',\
                                    'sd_limited','lact_limited']
metric = 'non_nan_avg'

#we use multiprocessing library to speed up the calculation
num_cores = multiprocessing.cpu_count()
st = time.time()
json_data_v = Parallel(n_jobs=num_cores)(delayed(save_mvg_agg)(file_n,keys,metric) for file_n in tqdm(glob.glob('trends_activity_level_second_level_data/*/*.json')))


df_agg = pd.concat([sublist for sublist in json_data_v])
print(len(set(df_agg['person_id'])))
ed = time.time()
print(ed-st)
df_agg.to_csv(f'{BUCKET}/{paths_dict.get("person_agg_summ_stats")}.tsv',sep='\t',index = False)

## Save a json with population level averages for all activity levels

This function processes the calculated trends at person_level to compute comprehensive temporal trend statistics across multiple time scales fro entire population. It aggregates individual participant JSON files to calculate population-level means, standard deviations, and sample sizes for daily, weekly, monthly, seasonal, and yearly activity patterns, enabling longitudinal analysis of physical activity trends.

In [ ]:

def save_plot_trends_arrays(bucket_name = bucket_.replace("gs://",""),\
                            folder_names = [f"jsons_calculations_{kkk}" for kkk in range(1,61)] , \
                            keys = ['mvg_limited','sd_limited','lact_limited',\
                                    'steps_limited'],\
                            metrics = ['non_nan_avg'],\
                            prefix_name = f'{paths_dict.get("trends_save")}',\
                           seasonal_hr = seasonal_hr,\
                           df_3_yearly_hr = df_3_yearly_hr,\
                           df_5_yearly_hr = df_5_yearly_hr):
    
    
    """
    Parameters:
    bucket_name: Google Cloud Storage bucket name (cleaned of 'gs://' prefix)
    folder_names: List of 60 folders containing JSON files (jsons_calculations_1 through jsons_calculations_60)
    keys: Activity metrics to analyze ('mvg_limited', 'sd_limited', 'lact_limited', 'steps_limited')
    metrics: Statistical measures to compute (default: 'non_nan_avg')
    prefix_name: name of the folder in gsc bucket where to read from
    seasonal_hr: Set of participant IDs to consider for seasonal analysis
    df_3_yearly_hr: Set of participant IDs with 3+ years of data
    df_5_yearly_hr: Set of participant IDs with 5+ years of data

    Saves results to trends_activity_averages.json
    """
    def nested_dict():
        return defaultdict(list)
    #Creates hierarchical data structures to organize activity metrics by time periods and participant eligibility
    data = defaultdict(nested_dict)
    data_seasonal = defaultdict(nested_dict)
    data_yearly_3 = defaultdict(nested_dict)
    data_yearly_5 = defaultdict(nested_dict)
    
    file_names = list_json_files(bucket_name, folder_names, prefix_name)
    for idx, file_n in enumerate(file_names):
        if idx % 10000 == 0:
            print(f"{idx} files processed")
        id___ = int(file_n.split('/')[-1].split('.')[0])
        json_data = read_json_file(bucket_name, file_n)
        for key in keys:
            for metric in metrics:
                # Extract data for each time period and append to the corresponding list
                try:
                    data[(key, metric)]['Daily'].append(json_data[f"daily_array_act_lvl"][key][metric])
                    data[(key, metric)]['Weekly'].append(json_data[f"weekly_stats_act_lvl"][key][metric])
                    data[(key, metric)]['Monthly'].append(json_data[f"monthly_stats_act_lvl"][key][metric])
                    
                    if id___ in seasonal_hr:
                        data_seasonal[(key, metric)]['Seasonal'].append(json_data[f"seasonal_stats_act_lvl"][key][metric])
                    if id___ in df_3_yearly_hr:
                        data_yearly_3[(key, metric)]['Yearly_3'].append(json_data[f"yearly_stats_act_lvl_3_gap"][key][metric][0])
                    if id___ in df_5_yearly_hr:
                        data_yearly_5[(key, metric)]['Yearly_5'].append(json_data[f"yearly_stats_act_lvl_3_gap"][key][metric][0])
                         
                except KeyError as e:
                    print(f"KeyError: {e} in file {file_n}")
    
    
    
    def process_data_dict(data_dict, save_data):
        """Helper function to process data dictionaries efficiently"""
        for (key, metric), periods in data_dict.items():
            for period, values in periods.items():
                if not values:  # Skip empty lists
                    continue
                    
                mean_vals = np.nanmean(values, axis=0)
                std_vals = np.nanstd(values, axis=0)
                n = np.sum(~np.isnan(values), axis=0)
                
                save_data[f'{period}_{key}_{metric}_mean'].append(list(mean_vals)) 
                save_data[f'{period}_{key}_{metric}_std'].append(list(std_vals)) 
                save_data[f'{period}_{key}_{metric}_n'].append([float(i) for i in list(n)])

    def process_yearly_data(data_dict, save_data):
        """Optimized yearly data processing"""
        for (key, metric), periods in data_dict.items():
            for period, values in periods.items():
                if not values:
                    continue
                    
                # Use defaultdict for faster aggregation
                yearly_values = defaultdict(list)
                for entry in values:
                    for year, value in entry.items():
                        yearly_values[year].append(value)
                
                # Vectorized operations
                years = sorted(yearly_values.keys(), key=lambda x: int(x.split()[1]))
                mean_vals = [np.mean(yearly_values[year]) for year in years]
                std_vals = [np.std(yearly_values[year]) for year in years]
                n = [np.sum(~np.isnan(yearly_values[year])) for year in years]
                
                save_data[f'{period}_{key}_{metric}_mean'].append(list(mean_vals)) 
                save_data[f'{period}_{key}_{metric}_std'].append(list(std_vals)) 
                save_data[f'{period}_{key}_{metric}_n'].append([float(i) for i in list(n)])  

    # Process all data types
    save_data = defaultdict(list)
    process_data_dict(data, save_data)
    process_data_dict(data_seasonal, save_data)
    process_yearly_data(data_yearly_3, save_data)
    process_yearly_data(data_yearly_5, save_data)

    # Save the trend data to a JSON file
    json_filename = f'{paths_dict.get("trends_save_avg")}'
    with open(json_filename, "w") as json_file:
        json.dump(save_data, json_file, indent=4)
    print(f"Saved {json_filename}")

# Example usage

st = time.time()
save_plot_trends_arrays()
et = time.time()
print(et-st)

## Plots figure 1 panel

This function creates comprehensive temporal trend visualizations across multiple time scales. It generates six different plots showing how a specific activity metric (like steps or movement) varies over daily, weekly, monthly, seasonal, and yearly patterns, using pre-computed population-level statistics with confidence intervals to reveal temporal activity patterns  

Expects a json file with saved metrics to plot

In [ ]:
def plot_save_trends_bar(key = 'steps_limited', metric = 'non_nan_avg', \
                         json_file_name = f'{paths_dict.get("trends_save_avg")}' ):
    
    """
    Parameters:
    key: Activity metric to visualize (default: 'steps_limited')
    metric: Statistical measure to plot (default: 'non_nan_avg')
    json_file_name: saved metrics to plot
    """
    
    
    timestamps = pd.date_range("00:00", "23:59", freq="1min").strftime('%H:%M')
    tick_indices = np.array(np.arange(0, 1440, 120).tolist()+[1439])
    tick_labels = [timestamps[i] for i in tick_indices]
    days_of_month = np.arange(0, 31)
    days_of_month_l = np.arange(1, 32)
    days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    month = np.arange(0, 12)
    x = np.arange(len(days))
    
    with open(f"{json_file_name}", "r") as json_file:
        trend_data = json.load(json_file)
    for kk,tickindices_jj,ticklabels_jj,labels_n in zip(['Daily','Weekly','Monthly','Seasonal','Yearly_3','Yearly_5'],\
                                                 [tick_indices,x,days_of_month, month,'',''],\
                                                 [tick_labels,days,days_of_month_l,months,'',''],\
                                                  ['Time of day','Weekday','Day of month','Month','Years','Years']):
    
        mean_vals = trend_data[f'{kk}_{key}_{metric}_mean'][0]
        std_vals = trend_data[f'{kk}_{key}_{metric}_std'][0]
        n = trend_data[f'{kk}_{key}_{metric}_n'][0]
        
        
        if 'Yearly' in kk:
            no_of_years = int(kk.split('_')[-1])
            
            years = [f'Year_{i+1}' for i in range(len(mean_vals))][0:no_of_years]  
            tickindices_jj = np.arange(len(years))[0:no_of_years]  
            ticklabels_jj = years[0:no_of_years]  
            ci_95 = 1.96 * (std_vals[0:no_of_years]   / np.sqrt(n[0:no_of_years]  ))

        else:
            
            ci_95 = 1.96 * (std_vals / np.sqrt(n))
        

        plt.figure(figsize=(10, 5))
        if kk=='Daily':
            plt.plot(mean_vals, label="Mean", color='blue')
            plt.fill_between(range(len(mean_vals)), mean_vals - ci_95, mean_vals + ci_95, color='red', alpha=0.2, label="95% CI")
        elif 'Yearly' in kk:
            plt.bar(range(len(mean_vals))[0:no_of_years]  , mean_vals[0:no_of_years]  , color='blue', alpha=0.4, label="Mean")
            plt.errorbar(range(len(mean_vals))[0:no_of_years]  , mean_vals[0:no_of_years]  , yerr=ci_95, fmt='None', ecolor='red', capsize=5, label="95% CI")
        else:
            plt.bar(range(len(mean_vals)), mean_vals, color='blue', alpha=0.4, label="Mean")
            plt.errorbar(range(len(mean_vals)), mean_vals, yerr=ci_95, fmt='None', ecolor='red', capsize=5, label="95% CI")

        plt.xlabel(labels_n, fontsize = 15)
        if 'mvg' in key:
            plt.ylabel(get_ylabel(key, kk), fontsize = 13)
        else:
            plt.ylabel(get_ylabel(key, kk), fontsize = 15)
        
        plt.title(f"{kk} Trends", fontsize = 15)
        plt.legend(fontsize = 15,loc = 'upper right')
        plt.xticks(tickindices_jj,labels = ticklabels_jj, fontsize = 12)
        if '1440' in key and ('Monthly' in kk or 'Weekly' in kk):
            plt.ylim(0,int(str(np.max(mean_vals))[0])*get_ycutoff(key, kk)[0]*10+get_ycutoff(key, kk)[1])
        elif kk!='Daily' and 'Yearly' not in kk:
            plt.ylim(0,int(str(np.max(mean_vals))[0])*get_ycutoff(key, kk)[0]+get_ycutoff(key, kk)[1])
        elif 'Yearly' in kk:
            plt.ylim(0,int(str(np.max(mean_vals[0:no_of_years]))[0])*get_ycutoff(key, kk)[0]+get_ycutoff(key, kk)[1])
        plt.show()
    

In [ ]:
# plot_save_trends_bar(key = 'steps_limited', metric = 'non_nan_avg',\
# json_file_name = f'{paths_dict.get("trends_save_avg")}'

## Socio-Demographic stratification save trends

This function processes wearable device activity data to compute stratified trend statistics across multiple demographic groups and time periods. It reads individual participant JSON files containing activity metrics, groups participants by demographics (age, race, sex, education, employment, income) and wearable consent status, then calculates aggregate statistics (mean, standard deviation, sample size) for daily, weekly, and monthly activity patterns  

The function loads participant demographics from all_person_stats.tsv and calculates group sizes for each demographic category:  

Age groups: 18-44, 45-65, 66+ years  
Sex: Female, Male  
Race: White, Black, Asian, Other  
Education: advanced degree,some_degree, no_degree  
Employment: Student, Retired, Job, Not working  
Income: >200K, 100K-200K, 50K-100K, <50K

In [ ]:
def save_plot_trends_arrays_stratified(bucket_name = bucket_.replace("gs://",""),\
                            folder_names = [f"jsons_calculations_{kkk}" for kkk in range(1,61)] , \
                            keys = ['mvg_limited','sd_limited','lact_limited',\
                                    'steps_limited'],metrics = ['non_nan_avg'],\
                                      prefix_name = f'{paths_dict.get("trends_save")}'):
    
    """
    Parameters:
    bucket_name: Google Cloud Storage bucket name (without 'gs://' prefix)
    folder_names: List of folder names containing JSON files 
    (defaults to 60 folders: jsons_calculations_1 through jsons_calculations_60)
    keys: Activity metrics to analyze (sedentary_minutes, steps, etc.)
    metrics: Statistical measures to compute (default: 'non_nan_avg')
    prefix_name: name of the folder in gsc bucket where to read from
    
    
    
    Saves results to trends_activity_stratified_data_averages_qc.json
    """
    
    
    df_summary = pd.read_csv(f'{bucket_}/{paths_dict.get("person_stats_file")}.tsv',sep='\t')
    wear_consent_ids = list(set(df_summary[df_summary['wear_consent']==1]['person_id']))
    
    groups = {
        'age': {
            '18-44': set(df_summary[(df_summary['age_at_cdr']>=18)&(df_summary['age_at_cdr']<45)]['person_id']),
            '45-65': set(df_summary[(df_summary['age_at_cdr']>=45)&(df_summary['age_at_cdr']<66)]['person_id']),
            '>65': set(df_summary[df_summary['age_at_cdr']>=66]['person_id'])
        },
        'race': {
            'white': set(df_summary[df_summary['Race']=='White']['person_id']),
            'black': set(df_summary[df_summary['Race']=='Black or African American']['person_id']),
            'asian': set(df_summary[df_summary['Race']=='Asian']['person_id']),
            'other': set(df_summary[df_summary['Race']=='Other']['person_id'])
        },
        'sex': {
            'female': set(df_summary[df_summary['sex_at_birth']=='Female']['person_id']),
            'male': set(df_summary[df_summary['sex_at_birth']=='Male']['person_id'])
        },
        'education': {
            'advanced': set(df_summary[df_summary['education_level']=='College Degree or Advanced']['person_id']),
            'some_degree': set(df_summary[df_summary['education_level']=='Some college degree']['person_id']),
            'no_degree': set(df_summary[df_summary['education_level']=='No college degree']['person_id'])
        },
        'employment': {
            'student': set(df_summary[df_summary['employment_type']=='Student']['person_id']),
            'retired': set(df_summary[df_summary['employment_type']=='Retired']['person_id']),
            'job': set(df_summary[(df_summary['employment_type']=='Self Employed')|(df_summary['employment_type']=='Job')]['person_id']),
            'not_working': set(df_summary[(df_summary['employment_type']=='Homemaker')|(df_summary['employment_type']=='Not working')]['person_id'])
        },
        'income': {
            'more_than_200k': set(df_summary[df_summary['income_level']=='>200K']['person_id']),
            '_100k_200k': set(df_summary[df_summary['income_level']=='100K-200K']['person_id']),
            '_50k_100k': set(df_summary[df_summary['income_level']=='50K-100K']['person_id']),
            'less_than_50k': set(df_summary[(df_summary['income_level']=='<10K')|(df_summary['income_level']=='10K-50K')]['person_id'])
        }
    }
  



    def nested_dict():
        return defaultdict(list)
    
    all_data = defaultdict(nested_dict)
    
    file_names = list_json_files(bucket_name, folder_names, prefix_name)
    
    for idx, file_n in enumerate(file_names):
        if idx % 10000 == 0:
            print(f"{idx} files processed")
            
        person_id = int(file_n.split('/')[-1].split('.')[0])
        json_data = read_json_file(bucket_name, file_n)
        
        is_wear_consent = person_id in wear_consent_ids
        
        person_groups = {}
        for type_group, group_dict in groups.items():
            for group_name, group_set in group_dict.items():
                if person_id in group_set:
                    person_groups[type_group] = group_name
                    break
        
        for key in keys:
            for metric in metrics:
                try:
                    daily_data = json_data[f"daily_array_act_lvl"][key][metric]
                    weekly_data = json_data[f"weekly_stats_act_lvl"][key][metric]
                    monthly_data = json_data[f"monthly_stats_act_lvl"][key][metric]
                    
                    for type_group, group_name in person_groups.items():
                        # All data
                        all_data[(key, metric, type_group, group_name, 'all')]['Daily'].append(daily_data)
                        all_data[(key, metric, type_group, group_name, 'all')]['Weekly'].append(weekly_data)
                        all_data[(key, metric, type_group, group_name, 'all')]['Monthly'].append(monthly_data)
                        
                        # Wear consent stratified data
                        consent_type = 'wear' if is_wear_consent else 'no_wear'
                        all_data[(key, metric, type_group, group_name, consent_type)]['Daily'].append(daily_data)
                        all_data[(key, metric, type_group, group_name, consent_type)]['Weekly'].append(weekly_data)
                        all_data[(key, metric, type_group, group_name, consent_type)]['Monthly'].append(monthly_data)
                        
                except KeyError as e:
                    print(f"KeyError: {e} in file {file_n}")
    
    save_data = defaultdict(list)
    
    for (key, metric, type_group, group_name, consent_type), periods in all_data.items():
        for period, values in periods.items():
            if not values:
                continue
                
            values_array = np.array(values)
            mean_vals = np.nanmean(values_array, axis=0)
            std_vals = np.nanstd(values_array, axis=0)
            n = np.sum(~np.isnan(values_array), axis=0)
            
            prefix = f'{period}_{key}_{metric}_{type_group}_{group_name}_{consent_type}'
            save_data[f'{prefix}_mean'].append(list(mean_vals))
            save_data[f'{prefix}_std'].append(list(std_vals))
            save_data[f'{prefix}_n'].append([float(i) for i in list(n)])
    
    # Save results
    json_filename = f'{paths_dict.get("trends_save_avg_stratify")}'
    with open(json_filename, "w") as json_file:
        json.dump(save_data, json_file)  # No indent for faster writing
    print(f"Saved {json_filename}")

    
st = time.time()
save_plot_trends_arrays_stratified()
et = time.time()
print(et-st)        


## Socio-Demographic stratification plots

This function creates stratified visualizations of activity trends (like step count) across different demographic groups and time patterns. It generates both daily (hourly patterns) and weekly (day-of-week patterns) trend plots, comparing different demographic strata (age, sex, race, education, employment, income) while also considering wear consent status  

The function loads participant demographics from all_person_stats.tsv and calculates group sizes for each demographic category:  

Age groups: 18-44, 45-65, 66+ years  
Sex: Female, Male  
Race: White, Black, Asian, Other  
Education: advanced degree,some_degree, no_degree  
Employment: Student, Retired, Job, Not working  
Income: >200K, 100K-200K, 50K-100K, <50K  

Expects a json file with saved metrics to plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import json
import pandas as pd

def plot_save_trends_stratified_bar(
    key,
    metric='non_nan_avg',
    stratify='age',
    group_names=['18-44', '45-65', '>65'],
    wear_consent_types = ['all','wear','no_wear'],
    json_file_name = f'{paths_dict.get("trends_save_avg_stratify")}'
):
    
    """
    Parameters:
    key: The activity metric to plot (e.g., 'steps', 'sedentary minutes')
    metric: Statistical measure to use ('non_nan_avg' for non-null averages)
    stratify: Demographic variable to stratify by ('age', 'sex', 'race', 'education', 'employment', 'income')
    group_names: Labels for the stratification groups (default: age groups)
    wear_consent_types: groups to plot for with/without considering wear consent
    json_file_name: saved metrics to plot
    """
    
    timestamps = pd.date_range("00:00", "23:59", freq="1min").strftime('%H:%M')
    tick_indices_daily = np.array(np.arange(0, 1440, 120).tolist() + [1439])
    tick_labels_daily = [timestamps[i] for i in tick_indices_daily]
    x_weekly = np.arange(7)
    labels_weekly = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    patterns = {
        'Daily': (tick_indices_daily, tick_labels_daily, 'Time of day'),
        'Weekly': (x_weekly, labels_weekly, 'Weekday')
    }
    df_summary_i = pd.read_csv(f'{bucket_}/{paths_dict.get("person_stats_file")}.tsv',sep='\t')
    n_groups = len(group_names)
    colors_list = ['green','blue','magenta','orange','brown','cyan','purple']
    if n_groups <= 3:
        nrows, ncols = 1, 1
    else:
        nrows, ncols = 1, 1
    for wear_consent_type in wear_consent_types:
        for kk, (tickindices_jj, ticklabels_jj, x_label) in patterns.items():
            if kk=='Weekly' and (stratify =='income' or stratify =='employment'):
                fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(10, 5 * nrows), sharey=True)
            else:
                fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(9 * ncols, 5 * nrows), sharey=True)

            with open(f"{json_file_name}", "r") as json_file:
                trend_data = json.load(json_file)

            for i, group_name in enumerate(group_names):
                if wear_consent_type == 'all':
                    title_add = 'all'
                    df_summary = df_summary_i[df_summary_i['wear_consent'].isin([0,1])]
                elif wear_consent_type == 'wear':
                    title_add = 'wear consent'
                    df_summary = df_summary_i[df_summary_i['wear_consent'].isin([1])]
                else:
                    title_add = 'non wear consent'
                    df_summary = df_summary_i[df_summary_i['wear_consent'].isin([0])]



                ages = [len(list(set(df_summary[(df_summary['age_at_cdr']>=18)&(df_summary['age_at_cdr']<45)]['person_id']))),\
                len(list(set(df_summary[(df_summary['age_at_cdr']>=45)&(df_summary['age_at_cdr']<66)]['person_id']))),\
                len(list(set(df_summary[(df_summary['age_at_cdr']>=66)]['person_id'])))]

                sexes = [len(list(set(df_summary[df_summary['sex_at_birth']=='Female']['person_id']))),\
                len(list(set(df_summary[df_summary['sex_at_birth']=='Male']['person_id'])))]

                races = [len(list(set(df_summary[df_summary['Race']=='White']['person_id']))),\
                len(list(set(df_summary[df_summary['Race']=='Black or African American']['person_id']))),\
                len(list(set(df_summary[df_summary['Race']=='Asian']['person_id']))),\
                len(list(set(df_summary[df_summary['Race']=='Other']['person_id'])))]

                educations = [len(list(set(df_summary[df_summary['education_level']=='College Degree or Advanced']['person_id']))),\
                len(list(set(df_summary[df_summary['education_level']=='Some college degree']['person_id']))),\
                len(list(set(df_summary[df_summary['education_level']=='No college degree']['person_id'])))]

                employments = [len(list(set(df_summary[df_summary['employment_type']=='Student']['person_id']))),\
                len(list(set(df_summary[df_summary['employment_type']=='Retired']['person_id']))),\
                len(list(set(df_summary[(df_summary['employment_type']=='Job')|\
                                       (df_summary['employment_type']=='Self Employed')]['person_id']))),\
                len(list(set(df_summary[(df_summary['employment_type']=='Not working')|\
                                       (df_summary['employment_type']=='Homemaker')]['person_id'])))]

                incomes = [len(list(set(df_summary[df_summary['income_level']=='>200K']['person_id']))),\
                len(list(set(df_summary[df_summary['income_level']=='100K-200K']['person_id']))),\
                len(list(set(df_summary[df_summary['income_level']=='50K-100K']['person_id']))),\
                len(list(set(df_summary[(df_summary['income_level']=='<10K')|\
                                       (df_summary['income_level']=='10K-50K')]['person_id'])))]

                no_dict = {'age':ages,'sex':sexes,'race':races, 'education':educations, \
                           'employment':employments,'income':incomes}
                lists_pick = no_dict[stratify]

                mean_vals = trend_data[f'{kk}_{key}_{metric}_{stratify}_{group_name}_{wear_consent_type}_mean'][0]
                std_vals = trend_data[f'{kk}_{key}_{metric}_{stratify}_{group_name}_{wear_consent_type}_std'][0]
                n = trend_data[f'{kk}_{key}_{metric}_{stratify}_{group_name}_{wear_consent_type}_n'][0]

                total_n = lists_pick[i]
                if len(mean_vals) == 0:
                    continue

                ci_95 = 1.96 * (std_vals / np.sqrt(n))
                ax = axs

                if kk == 'Daily':
                    ax.plot(mean_vals, label=f" {group_name} (n={total_n})", color=colors_list[i])
                    ax.fill_between(range(len(mean_vals)), mean_vals - ci_95, mean_vals + ci_95,
                                    color='red', alpha=0.3)
                else:
                    if kk=='Weekly' and (stratify =='income' or stratify =='employment'):
                        bar_width = 0.1
                    else:
                        bar_width = 0.2
                    x = np.arange(len(mean_vals))
                    offset = (i - n_groups / 2) * bar_width + bar_width / 2

                    ax.bar(x + offset, mean_vals, width=bar_width, color=colors_list[i], alpha=0.6, label=f" {group_name} (n={total_n})")
                    ax.errorbar(x + offset, mean_vals, yerr=ci_95, fmt='none', ecolor='black', capsize=3)



                ax.set_xlabel(x_label, fontsize = 15)
                ax.set_xticks(tickindices_jj)
                ax.set_xticklabels(ticklabels_jj, rotation=45, fontsize = 15)
                ax.set_title(f"{kk} Trends by {stratify.title()} - {title_add}", fontsize=16)
                if i == 0:
                    ax.set_ylabel("Average Step Count", fontsize = 15)
                ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., fontsize = 15)
            plt.tight_layout(rect=[0, 0, 1, 0.95])

            plt.show()


In [ ]:
groups_lists = {
        'age': ['18-44','45-65','>65'],
        'race': ['white','black','asian','other'],
        'sex': ['female','male'],
        'education': ['advanced','some_degree','no_degree'],
        'employment': ['student','retired','job','not_working'],
        'income': ['more_than_200k','_100k_200k','_50k_100k','less_than_50k']
    }

In [ ]:
stratify_key = list(groups_lists.keys())[0]

plot_save_trends_stratified_bar(key = 'steps_limited', metric = 'non_nan_avg',stratify = stratify_key,\
                                group_names = groups_lists[stratify_key],\
                                wear_consent_types = ['all','wear','no_wear'],
    json_file_name = f'{paths_dict.get("trends_save_avg_stratify")}')